In [1]:
# ============================================================
# SCORE CIGA BASE — Palmeiras
# Posições vindas da própria base de dados
# Últimos 5 jogos do Brasileirão, lidos da planilha
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# ETAPA 1 — Carregar a base da planilha
# Cada linha = um jogador x jogo
# A posição já vem preenchida na base
# ============================================================

ARQUIVO_BASE = Path('palmeiras_ultimos_5_jogos_brasileirao.xlsx')

COLUNAS_MODELO = [
    'Jogador', 'Posição', 'Nível do jogo', 'Minutos jogados',
    'Golos', 'xG', 'Assistências', 'xA',
    'Remates', 'Remates no alvo',
    'Passes', 'Passes certos',
    'Cruzamentos', 'Cruzamentos certos',
    'Dribles', 'Dribles certos',
    'Duelos', 'Duelos certos',
    'Perdas (total)', 'Recuperações (total)',
    'Toques na área',
    'Cartões amarelos', 'Cartões vermelhos',
    'Duelos defensivos', 'Duelos ofensivos', 'Duelos aéreos',
    'Remates bloqueados', 'Interceções', 'Alívios (Clearances)',
    'Faltas cometidas', 'Faltas sofridas',
    'Passes decisivos', 'Segunda assistência', 'Terceira assistência',
    'Shot assists',
]

df = pd.read_excel(ARQUIVO_BASE, sheet_name='dados_brutos')
df = df[COLUNAS_MODELO].copy()

# O modelo de pesos não contempla goleiro
df = df[df['Posição'] != 'Goleiro'].copy()

print('Base carregada:', df.shape)
df.head()

# ============================================================
# ETAPA 2 — Pesos OFENSIVOS por posição
# O peso muda conforme a função tática do jogador
# ============================================================

pesos_ofensivos = {
    'Zagueiro': {
        'Golos': 0.07, 'xG': 0.05, 'Assistências': 0.05, 'xA': 0.04,
        'Remates': 0.03, 'Remates no alvo': 0.04,
        'Passes': 0.10, 'Passes certos': 0.15,
        'Cruzamentos': 0.01, 'Cruzamentos certos': 0.02,
        'Dribles': 0.02, 'Dribles certos': 0.03,
        'Duelos ofensivos': 0.05, 'Toques na área': 0.04, 'Faltas sofridas': 0.03,
        'Passes decisivos': 0.08, 'Segunda assistência': 0.06,
        'Terceira assistência': 0.07, 'Shot assists': 0.06,
    },
    'Lateral Direito': {
        'Golos': 0.05, 'xG': 0.04, 'Assistências': 0.10, 'xA': 0.08,
        'Remates': 0.03, 'Remates no alvo': 0.03,
        'Passes': 0.05, 'Passes certos': 0.07,
        'Cruzamentos': 0.07, 'Cruzamentos certos': 0.10,
        'Dribles': 0.03, 'Dribles certos': 0.05,
        'Duelos ofensivos': 0.04, 'Toques na área': 0.03, 'Faltas sofridas': 0.03,
        'Passes decisivos': 0.09, 'Segunda assistência': 0.05,
        'Terceira assistência': 0.02, 'Shot assists': 0.04,
    },
    'Lateral Esquerdo': {
        'Golos': 0.05, 'xG': 0.04, 'Assistências': 0.10, 'xA': 0.08,
        'Remates': 0.03, 'Remates no alvo': 0.03,
        'Passes': 0.05, 'Passes certos': 0.07,
        'Cruzamentos': 0.07, 'Cruzamentos certos': 0.10,
        'Dribles': 0.03, 'Dribles certos': 0.05,
        'Duelos ofensivos': 0.04, 'Toques na área': 0.03, 'Faltas sofridas': 0.03,
        'Passes decisivos': 0.09, 'Segunda assistência': 0.05,
        'Terceira assistência': 0.02, 'Shot assists': 0.04,
    },
    'Volante': {
        'Golos': 0.04, 'xG': 0.03, 'Assistências': 0.07, 'xA': 0.06,
        'Remates': 0.03, 'Remates no alvo': 0.03,
        'Passes': 0.08, 'Passes certos': 0.12,
        'Cruzamentos': 0.02, 'Cruzamentos certos': 0.02,
        'Dribles': 0.02, 'Dribles certos': 0.04,
        'Duelos ofensivos': 0.04, 'Toques na área': 0.03, 'Faltas sofridas': 0.04,
        'Passes decisivos': 0.11, 'Segunda assistência': 0.10,
        'Terceira assistência': 0.06, 'Shot assists': 0.06,
    },
    'Meia': {
        'Golos': 0.08, 'xG': 0.06, 'Assistências': 0.11, 'xA': 0.09,
        'Remates': 0.04, 'Remates no alvo': 0.05,
        'Passes': 0.05, 'Passes certos': 0.07,
        'Cruzamentos': 0.02, 'Cruzamentos certos': 0.03,
        'Dribles': 0.04, 'Dribles certos': 0.06,
        'Duelos ofensivos': 0.04, 'Toques na área': 0.04, 'Faltas sofridas': 0.04,
        'Passes decisivos': 0.11, 'Segunda assistência': 0.05,
        'Terceira assistência': 0.02, 'Shot assists': 0.03,
    },
    'Extremo': {
        'Golos': 0.11, 'xG': 0.08, 'Assistências': 0.09, 'xA': 0.07,
        'Remates': 0.05, 'Remates no alvo': 0.06,
        'Passes': 0.02, 'Passes certos': 0.03,
        'Cruzamentos': 0.04, 'Cruzamentos certos': 0.06,
        'Dribles': 0.05, 'Dribles certos': 0.08,
        'Duelos ofensivos': 0.05, 'Toques na área': 0.06, 'Faltas sofridas': 0.04,
        'Passes decisivos': 0.06, 'Segunda assistência': 0.02,
        'Terceira assistência': 0.01, 'Shot assists': 0.02,
    },
    'Centroavante': {
        'Golos': 0.16, 'xG': 0.12, 'Assistências': 0.06, 'xA': 0.05,
        'Remates': 0.06, 'Remates no alvo': 0.08,
        'Passes': 0.02, 'Passes certos': 0.04,
        'Cruzamentos': 0.01, 'Cruzamentos certos': 0.02,
        'Dribles': 0.02, 'Dribles certos': 0.04,
        'Duelos ofensivos': 0.05, 'Toques na área': 0.10, 'Faltas sofridas': 0.05,
        'Passes decisivos': 0.05, 'Segunda assistência': 0.02,
        'Terceira assistência': 0.01, 'Shot assists': 0.04,
    },
}

# ============================================================
# ETAPA 3 — Pesos DEFENSIVOS por posição
# Pesos NEGATIVOS penalizam ações indesejadas
# (perdas de bola, faltas e cartões)
# ============================================================

pesos_defensivos = {
    'Zagueiro': {
        'Recuperações (total)': 0.18, 'Duelos defensivos': 0.14,
        'Duelos': 0.06, 'Duelos certos': 0.10,
        'Interceções': 0.14, 'Alívios (Clearances)': 0.13,
        'Remates bloqueados': 0.13, 'Duelos aéreos': 0.12,
        'Perdas (total)': -0.05, 'Faltas cometidas': -0.04,
        'Cartões amarelos': -0.06, 'Cartões vermelhos': -0.15,
    },
    'Lateral Direito': {
        'Recuperações (total)': 0.18, 'Duelos defensivos': 0.15,
        'Duelos': 0.07, 'Duelos certos': 0.11,
        'Interceções': 0.15, 'Alívios (Clearances)': 0.11,
        'Remates bloqueados': 0.11, 'Duelos aéreos': 0.12,
        'Perdas (total)': -0.05, 'Faltas cometidas': -0.04,
        'Cartões amarelos': -0.06, 'Cartões vermelhos': -0.15,
    },
    'Lateral Esquerdo': {
        'Recuperações (total)': 0.18, 'Duelos defensivos': 0.15,
        'Duelos': 0.07, 'Duelos certos': 0.11,
        'Interceções': 0.15, 'Alívios (Clearances)': 0.11,
        'Remates bloqueados': 0.11, 'Duelos aéreos': 0.12,
        'Perdas (total)': -0.05, 'Faltas cometidas': -0.04,
        'Cartões amarelos': -0.06, 'Cartões vermelhos': -0.15,
    },
    'Volante': {
        'Recuperações (total)': 0.18, 'Duelos defensivos': 0.16,
        'Duelos': 0.08, 'Duelos certos': 0.12,
        'Interceções': 0.16, 'Alívios (Clearances)': 0.08,
        'Remates bloqueados': 0.11, 'Duelos aéreos': 0.11,
        'Perdas (total)': -0.06, 'Faltas cometidas': -0.05,
        'Cartões amarelos': -0.06, 'Cartões vermelhos': -0.15,
    },
    'Meia': {
        'Recuperações (total)': 0.19, 'Duelos defensivos': 0.15,
        'Duelos': 0.09, 'Duelos certos': 0.13,
        'Interceções': 0.15, 'Alívios (Clearances)': 0.06,
        'Remates bloqueados': 0.10, 'Duelos aéreos': 0.11,
        'Perdas (total)': -0.06, 'Faltas cometidas': -0.05,
        'Cartões amarelos': -0.06, 'Cartões vermelhos': -0.15,
    },
    'Extremo': {
        'Recuperações (total)': 0.20, 'Duelos defensivos': 0.17,
        'Duelos': 0.10, 'Duelos certos': 0.14,
        'Interceções': 0.15, 'Alívios (Clearances)': 0.04,
        'Remates bloqueados': 0.10, 'Duelos aéreos': 0.10,
        'Perdas (total)': -0.06, 'Faltas cometidas': -0.05,
        'Cartões amarelos': -0.06, 'Cartões vermelhos': -0.15,
    },
    'Centroavante': {
        'Recuperações (total)': 0.20, 'Duelos defensivos': 0.17,
        'Duelos': 0.11, 'Duelos certos': 0.15,
        'Interceções': 0.13, 'Alívios (Clearances)': 0.04,
        'Remates bloqueados': 0.08, 'Duelos aéreos': 0.12,
        'Perdas (total)': -0.06, 'Faltas cometidas': -0.05,
        'Cartões amarelos': -0.06, 'Cartões vermelhos': -0.15,
    },
}

# ============================================================
# ETAPA 4 — Consolidar por jogador
# A base tem 1 linha por jogador POR JOGO.
# Para ranquear o elenco, somamos as métricas de todos os jogos
# e usamos a média do nível dos adversários enfrentados.
# ============================================================

metricas = [c for c in df.columns
            if c not in ['Jogador', 'Posição', 'Nível do jogo', 'Minutos jogados']]

agregacao = {m: 'sum' for m in metricas}
agregacao['Minutos jogados'] = 'sum'
agregacao['Nível do jogo'] = 'mean'

df = df.groupby(['Jogador', 'Posição'], as_index=False).agg(agregacao)

print('Base consolidada por jogador:', df.shape)
df[['Jogador', 'Posição', 'Minutos jogados', 'Nível do jogo']]

# ============================================================
# ETAPA 5 — Ajuste de minutos (mínimo de 20)
# ============================================================

df['Minutos_Ajustados'] = df['Minutos jogados'].clip(lower=20)

df[['Jogador', 'Posição', 'Minutos jogados', 'Minutos_Ajustados']].head()

# ============================================================
# ETAPA 6 — Score Bruto
# "O que ele fez, ponderado pela posição"
# ============================================================

def calcular_score_bruto(row, pesos_dict):
    posicao = row['Posição']
    if posicao not in pesos_dict:
        return np.nan
    score = 0
    for variavel, peso in pesos_dict[posicao].items():
        valor = row.get(variavel, 0)
        valor = 0 if pd.isna(valor) else valor
        score += valor * peso
    return score

df['Score_Of_Bruto'] = df.apply(lambda row: calcular_score_bruto(row, pesos_ofensivos), axis=1)
df['Score_Def_Bruto'] = df.apply(lambda row: calcular_score_bruto(row, pesos_defensivos), axis=1)

df[['Jogador', 'Posição', 'Score_Of_Bruto', 'Score_Def_Bruto']].head(10)

# ============================================================
# ETAPA 7 — Score Ajustado (nível do jogo + minutos)
# "Quanto isso vale considerando o contexto"
# ============================================================

df['Score_Of_Ajustado'] = (df['Score_Of_Bruto'] * df['Nível do jogo']) / df['Minutos_Ajustados']
df['Score_Def_Ajustado'] = (df['Score_Def_Bruto'] * df['Nível do jogo']) / df['Minutos_Ajustados']

df[['Jogador', 'Posição', 'Score_Of_Ajustado', 'Score_Def_Ajustado']].head(10)

# ============================================================
# ETAPA 8 — Normalização 0-5 contra TODO o elenco
# "Comparação com todos os atletas, sem separar por posição"
# ============================================================

def normalizar_0_5(serie):
    min_val = serie.min()
    max_val = serie.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series([2.5] * len(serie), index=serie.index)
    return 5 * (serie - min_val) / (max_val - min_val)

df['Nota_Ofensiva_0_5'] = normalizar_0_5(df['Score_Of_Ajustado'])
df['Nota_Defensiva_0_5'] = normalizar_0_5(df['Score_Def_Ajustado'])

df[['Jogador', 'Posição', 'Nota_Ofensiva_0_5', 'Nota_Defensiva_0_5']].head(10)

# ============================================================
# ETAPA 9 — Ranking por posição
# ============================================================

ranking_of = df[['Jogador', 'Posição', 'Nota_Ofensiva_0_5']].copy()
ranking_of['Rank_Ofensivo'] = ranking_of.groupby('Posição')['Nota_Ofensiva_0_5'].rank(
    ascending=False, method='dense'
)

ranking_def = df[['Jogador', 'Posição', 'Nota_Defensiva_0_5']].copy()
ranking_def['Rank_Defensivo'] = ranking_def.groupby('Posição')['Nota_Defensiva_0_5'].rank(
    ascending=False, method='dense'
)

ranking_final = ranking_of.merge(ranking_def, on=['Jogador', 'Posição'])
ranking_final.sort_values(['Posição', 'Rank_Ofensivo'])

# ============================================================
# ETAPA 10 — Score Final (média das notas ofensiva e defensiva)
# ============================================================

ranking_final['Score_Final'] = (
    (ranking_final['Nota_Ofensiva_0_5'] + ranking_final['Nota_Defensiva_0_5']) / 2
).round(2)

# Ordenar do maior para o menor Score_Final (ranking geral)
ranking_final = ranking_final.sort_values('Score_Final', ascending=False).reset_index(drop=True)

ranking_final[['Jogador', 'Posição', 'Nota_Ofensiva_0_5', 'Nota_Defensiva_0_5', 'Score_Final']]

# ============================================================
# ETAPA 11 — Exportar para Excel com gráfico na aba Ranking
# ============================================================

from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.drawing.fill import PatternFillProperties, ColorChoice

nome_saida = 'resultado_scores_palmeiras.xlsx'

with pd.ExcelWriter(nome_saida, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Base_com_Scores', index=False)
    ranking_final.to_excel(writer, sheet_name='Ranking', index=False)

    workbook = writer.book
    aba_ranking = writer.sheets['Ranking']

    # Localizar as colunas Jogador e Score_Final
    cabecalhos = [c.value for c in aba_ranking[1]]
    col_jogador = cabecalhos.index('Jogador') + 1
    col_score = cabecalhos.index('Score_Final') + 1
    n_linhas = len(ranking_final)

    # Criar gráfico de barras horizontais
    grafico = BarChart()
    grafico.type = 'bar'          # 'bar' = barras horizontais
    grafico.grouping = 'clustered'
    grafico.title = 'Score Final por Atleta (ordem decrescente)'
    grafico.height = 16
    grafico.width = 26
    grafico.gapWidth = 60         # barras mais grossas, menos espaço entre elas

    # Em gráfico horizontal:
    #   x_axis = eixo das CATEGORIAS (nomes dos atletas)
    #   y_axis = eixo dos VALORES (score)
    grafico.x_axis.title = 'Atleta'
    grafico.y_axis.title = 'Score Final (0-5)'

    # Garantir que os dois eixos apareçam (LibreOffice esconde se delete != False)
    grafico.x_axis.delete = False
    grafico.y_axis.delete = False

    # Escala do eixo de VALORES fixa de 0 a 5
    grafico.y_axis.scaling.min = 0
    grafico.y_axis.scaling.max = 5

    dados_grafico = Reference(aba_ranking, min_col=col_score,
                              min_row=1, max_row=n_linhas + 1)
    categorias = Reference(aba_ranking, min_col=col_jogador,
                           min_row=2, max_row=n_linhas + 1)

    grafico.add_data(dados_grafico, titles_from_data=True)
    grafico.set_categories(categorias)

    # Cor única para todas as barras (mesma série)
    serie = grafico.series[0]
    serie.graphicalProperties.solidFill = '2E5C8A'
    serie.graphicalProperties.line.solidFill = '2E5C8A'

    # Rótulos de dados: mostrar SOMENTE o valor
    grafico.dataLabels = DataLabelList()
    grafico.dataLabels.showVal = True
    grafico.dataLabels.showSerName = False
    grafico.dataLabels.showCatName = False
    grafico.dataLabels.showLegendKey = False
    grafico.dataLabels.numFmt = '0.00'

    grafico.legend = None

    aba_ranking.add_chart(grafico, 'J2')

try:
    from google.colab import files
    files.download(nome_saida)
except ImportError:
    pass
print('Arquivo exportado:', nome_saida)


Base carregada: (72, 35)
Base consolidada por jogador: (19, 35)
Arquivo exportado: resultado_scores_palmeiras.xlsx


In [ ]:
# Abrir o dashboard HTML no navegador
from pathlib import Path
import webbrowser

html = Path("palmeiras_scores.html").resolve()
print("Abrindo:", html)
webbrowser.open(html.as_uri())
